# PK composability — dose → exposure → tumor dynamics → survival

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

Onkos *consumes* exposure; it does not model PK (that is Hypnos). The small `onkos.pk` bridge turns a dose/regimen — or an external Hypnos PK profile — into the exposure metric the exposure-response kernels expect, so the spec's headline composability claim runs self-contained: **PK → exposure → tumor-dynamics → survival, one open tier-annotated chain.** The PK generators here are illustrative; for real PK, feed a Hypnos profile via `pk.from_profile`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos import pk

ds = onkos.load()
pkargs = dict(ka=0.5, ke=0.05, v=5.0, f=0.8)
# Cornerstone exposure relation: C_avg = F*Dose/(CL*tau), CL = ke*V
m = pk.steady_state_metrics(dose=1200, tau=24, **pkargs)
print({k: round(v, 2) for k, v in m.items()})
assert m['c_avg'] == pkargs['f'] * 1200 / (pkargs['ke'] * pkargs['v'] * 24)

In [ ]:
# The full chain: dose -> C_avg -> Emax ER -> kill -> tumor -> OS.
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
er = 'exposure_response.emax_generic'
t = np.linspace(0, 104, 209)
for dose in [600, 1200, 2400]:
    cavg = pk.steady_state_metrics(dose=dose, tau=24, **pkargs)['c_avg']
    tr = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, exposure=cavg, exposure_response=er, t=t)
    print(f'dose {dose:>4} mg -> C_avg {cavg:6.1f} ug/L -> depth {tr.metrics["depth_of_response"]:.3f}, median OS {tr.median_os:.1f}')
    plt.plot(t, tr.os_curve, label=f'{dose} mg')
plt.axhline(0.5, ls=':', color='grey'); plt.legend(); plt.xlabel('weeks'); plt.ylabel('OS'); plt.title('dose -> OS via the PK chain');

In [ ]:
# Higher dose -> more exposure -> deeper response -> longer OS (the go/no-go chain).
def med_os(dose):
    cavg = pk.steady_state_metrics(dose=dose, tau=24, **pkargs)['c_avg']
    return onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, exposure=cavg, exposure_response=er, t=t).median_os
assert med_os(2400) > med_os(600)

# Ingest an external (Hypnos) concentration-time profile instead of generating one.
C = pk.from_profile([0, 8, 52, 104], [0, 300, 220, 140], t)
tr = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, exposure=C, exposure_response=er, t=t)
print('ingested Hypnos-style profile -> week8', round(tr.metrics['week8_relative_change'], 3), '| median OS', round(tr.median_os, 1))
assert tr.tumor_size.shape == t.shape